In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler

# 1. Load Data

# Cargar parquet con Name column
DATA_PATH = "../datasets/"
df = pd.read_parquet(DATA_PATH +'fund_esg_2024.parquet')

In [ ]:
#Looking at top 10 market value funds in Spain
df[df['Country'].str.contains('Spain')].sort_values(by='Market_Value_USD', ascending=False).head(10)

,Industry,Region,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,ESG_any
8510,Utilities,Europe,Spain,iberdrola sa,2392104663,2.73,2.73,0.186034,1,1,1,1,1
2987,Financials,Europe,Spain,banco bilbao vizcaya argentaria sa,1406968905,2.49,2.49,0.109420,1,1,1,1,1
1244,Consumer Discretionary,Europe,Spain,industria de diseno textil sa,1326902048,0.83,0.83,0.103193,1,1,1,0,1
3002,Financials,Europe,Spain,banco santander sa,1252296868,1.79,1.79,0.097391,1,1,1,1,1
7157,Technology,Europe,Spain,amadeus it group sa,813691936,2.56,2.56,0.063281,0,0,0,0,0
2810,Energy,Europe,Spain,repsol sa,685976449,4.81,4.81,0.053348,1,1,1,1,1
8198,Telecommunications,Europe,Spain,cellnex telecom sa,456360032,2.04,2.04,0.035491,0,0,1,0,1
5220,Industrials,Europe,Spain,ferrovial se,436541010,1.42,1.42,0.033950,0,0,0,0,0
8341,Telecommunications,Europe,Spain,telefonica sa,371184816,1.61,1.61,0.028867,0,1,1,0,1
4782,Industrials,Europe,Spain,aena sme sa,269079639,0.88,0.88,0.020926,0,0,0,0,0


Banco Santander has been chosen due to the researcher's acquaitance with it and its relevance in Latin America

In [6]:
df_test = df[df['Name'] == 'banco santander sa']

Toca: 1. hacer el  onehot de regiones e industrias, luego sí borrar las 3 columnas innecesarias
espera, creo que con el ID, el 3002, puedo buscarlo en esg_2024. que es el de preprocess completo. BUSCALOOO Y PRUBALOOO (que coincidan las cosas)

In [7]:
df_test

,Industry,Region,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,ESG_any
3002,Financials,Europe,Spain,banco santander sa,1252296868,1.79,1.79,0.097391,1,1,1,1,1


In [9]:
df_test.drop(['Name', 'Country', 'Voting'], axis=1)

,Industry,Region,Market_Value_USD,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,ESG_any
3002,Financials,Europe,1252296868,1.79,0.097391,1,1,1,1,1


In [16]:
# Finding the company in the preprocessed dataset
df_final = pd.read_parquet(DATA_PATH +'esg_2024')

In [17]:
df_final.head(2)

,Environmental,Social,Governance,Climate_change,ESG_any,Region_Africa,Region_Asia,Region_Europe,Region_Latin America,Region_Middle East,...,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Ownership_log_scaled,Market_Value_USD_log_scaled,Portfolio_Weight_scaled
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0.117371,0.638404,0.000179
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0.307686,0.588395,0.000054


In [12]:
df_final.shape

(8369, 26)

In [20]:
mask = (df['Ownership']== 0)
df[mask].shape

(276, 13)

In [22]:
df = df[df['Ownership'] != 0]

#Shape has to be 8369 rows now
df.shape

(8369, 13)

In [25]:
df[df['Name'].str.contains('banco santander sa')]

,Industry,Region,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,ESG_any
3002,Financials,Europe,Spain,banco santander sa,1252296868,1.79,1.79,0.097391,1,1,1,1,1


In [26]:
df_final.iloc[2999:3006]

,Environmental,Social,Governance,Climate_change,ESG_any,Region_Africa,Region_Asia,Region_Europe,Region_Latin America,Region_Middle East,...,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Ownership_log_scaled,Market_Value_USD_log_scaled,Portfolio_Weight_scaled
2999,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0.106814,0.580917,0.000045
3000,0,0,1,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0.063021,0.707139,0.000922
3001,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0.029001,0.492147,0.000005
3002,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0.198920,0.786573,0.006139
3003,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0.220431,0.700333,0.000784
3004,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0.252796,0.783761,0.005741
3005,1,1,0,1,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0.363692,0.723118,0.001350


In [28]:
def one_hot_encode(df, columns):
    """
    One-Hot Encode the specified categorical columns in a dataframe.

    Parameters:
    df (pd.DataFrame): Input dataframe
    columns (list): List of categorical columns to encode

    Returns:
    pd.DataFrame: DataFrame with one-hot encoded columns
    """
    for col in columns:
        # Create temporary one-hot encoded columns for the current categorical column
        # The new columns are prefixed with the original column name
        dummies = pd.get_dummies(df[col], prefix=col)

        #  Convert boolean columns to integers (0/1)
        dummies = dummies.astype(int)

        # Concatenate the new dummy columns horizontally to the original dataframe
        # axis=1 ensures we are adding columns, not rows
        df = pd.concat([df, dummies], axis=1)

        # Drop the original categorical column, since it is now represented by the dummy columns
        #    axis=1 specifies that we're dropping
        df.drop(col, axis=1, inplace=True)

    return df


In [30]:
df = one_hot_encode(df, columns= ['Region', 'Industry'])

In [31]:
df.head(2)

,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,...,Industry_Consumer Discretionary,Industry_Consumer Staples,Industry_Energy,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities
0,India,aarti industries ltd,8265429,0.48,0.48,0.000643,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Lithuania,ab grigeo,2506003,1.75,1.75,0.000195,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [33]:
df[df['Name'].str.contains('banco santander sa')]

,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,...,Industry_Consumer Discretionary,Industry_Consumer Staples,Industry_Energy,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities
3002,Spain,banco santander sa,1252296868,1.79,1.79,0.097391,1,1,1,1,...,0,0,0,1,0,0,0,0,0,0


In [35]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

columns = ['Portfolio_Weight', 'Ownership', 'Market_Value_USD']

for col in df[columns]:
    # Apply log transformation to compress the scale
    df[f'{col}_log'] = np.log1p(df[col])  # log1p to handle zero values safely

columns_log = [f'{col}_log' for col in ['Portfolio_Weight', 'Ownership', 'Market_Value_USD']]
for col in df[columns_log]:
    df[f'{col}_scaled']= scaler.fit_transform(df[[col]])
df.head(2)

,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,...,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Portfolio_Weight_log,Ownership_log,Market_Value_USD_log,Portfolio_Weight_log_scaled,Ownership_log_scaled,Market_Value_USD_log_scaled
0,India,aarti industries ltd,8265429,0.48,0.48,0.000643,0,0,0,0,...,0,0,0,0,0.000643,0.392042,15.927592,0.000421,0.117371,0.638404
1,Lithuania,ab grigeo,2506003,1.75,1.75,0.000195,0,0,0,0,...,0,0,0,0,0.000195,1.011601,14.734200,0.000128,0.307686,0.588395


In [43]:
df['Portfolio_Weight_scaled'] = scaler.fit_transform(df[['Portfolio_Weight']])

In [46]:
df_final.columns

Index(['Environmental', 'Social', 'Governance', 'Climate_change', 'ESG_any',
       'Region_Africa', 'Region_Asia', 'Region_Europe', 'Region_Latin America',
       'Region_Middle East', 'Region_North America', 'Region_Oceania',
       'Industry_Basic Materials', 'Industry_Consumer Discretionary',
       'Industry_Consumer Staples', 'Industry_Energy', 'Industry_Financials',
       'Industry_Health Care', 'Industry_Industrials', 'Industry_Real Estate',
       'Industry_Technology', 'Industry_Telecommunications',
       'Industry_Utilities', 'Ownership_log_scaled',
       'Market_Value_USD_log_scaled', 'Portfolio_Weight_scaled'],
      dtype='object')

In [44]:
df[df['Name'].str.contains('banco santander sa')]

,Country,Name,Market_Value_USD,Voting,Ownership,Portfolio_Weight,Environmental,Social,Governance,Climate_change,...,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Portfolio_Weight_log,Ownership_log,Market_Value_USD_log,Portfolio_Weight_log_scaled,Ownership_log_scaled,Market_Value_USD_log_scaled,Portfolio_Weight_scaled
3002,Spain,banco santander sa,1252296868,1.79,1.79,0.097391,1,1,1,1,...,0,0,0,0.092936,1.026042,20.948245,0.060953,0.312122,0.848796,0.0271


In [48]:
df.drop(['Country', 'Voting', 'Market_Value_USD_log', 'Portfolio_Weight_log_scaled', 'Portfolio_Weight_log', 'Ownership_log', 'Country', 'Ownership'], axis=1, inplace=True)

In [50]:
df.drop(['Market_Value_USD', 'Portfolio_Weight'], axis=1, inplace=True)

In [51]:
df.head(2)

,Name,Environmental,Social,Governance,Climate_change,ESG_any,Region_Africa,Region_Asia,Region_Europe,Region_Latin America,...,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Ownership_log_scaled,Market_Value_USD_log_scaled,Portfolio_Weight_scaled
0,aarti industries ltd,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0.117371,0.638404,0.000179
1,ab grigeo,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0.307686,0.588395,0.000054


In [54]:
santander= df[df['Name'].str.contains('banco santander sa')]
santander

,Name,Environmental,Social,Governance,Climate_change,ESG_any,Region_Africa,Region_Asia,Region_Europe,Region_Latin America,...,Industry_Financials,Industry_Health Care,Industry_Industrials,Industry_Real Estate,Industry_Technology,Industry_Telecommunications,Industry_Utilities,Ownership_log_scaled,Market_Value_USD_log_scaled,Portfolio_Weight_scaled
3002,banco santander sa,1,1,1,1,1,0,0,1,0,...,1,0,0,0,0,0,0,0.312122,0.848796,0.0271


In [58]:
import joblib

# santander dataframe already exists with your data

# Prepare features - drop Name and target columns
target_columns = ['Name', 'Environmental', 'Social', 'Governance', 'Climate_change', 'ESG_any']
X_santander = santander.drop(columns=target_columns)

# Store actual labels
actuals = {
    'ESG_any': santander['ESG_any'].values[0],
    'Governance': santander['Governance'].values[0],
    'Social': santander['Social'].values[0],
    'Environmental': santander['Environmental'].values[0],
    'Climate_change': santander['Climate_change'].values[0]
}

print("="*70)
print("BANCO SANTANDER SA - PREDICTIONS")
print("="*70)

# Model paths
models = {
    'ESG_any': '../models/models_esg_any/XGBoost.pkl',
    'Governance': '../models/models_governance/CatBoost.pkl',
    'Social': '../models/models_social/CatBoost.pkl',
    'Environmental': '../models/models_environmental/CatBoost.pkl',
    'Climate_change': '../models/models_climate_change/XGBoost.pkl'
}

results = {}

for target, model_path in models.items():
    model = joblib.load(model_path)
    
    pred = model.predict(X_santander)[0]
    proba = model.predict_proba(X_santander)[0]
    
    results[target] = {
        'pred': pred,
        'actual': actuals[target],
        'prob_yes': proba[1]
    }
    
    match = '✓' if pred == actuals[target] else '✗'
    
    print(f"\n{target}:")
    print(f"  Prediction: {pred} | Actual: {actuals[target]} | {match}")
    print(f"  Probability: {proba[1]:.2%}")

correct = sum([1 for r in results.values() if r['pred'] == r['actual']])
print(f"\n\nAccuracy: {correct}/5 ({correct/5*100:.0f}%)")

BANCO SANTANDER SA - PREDICTIONS

ESG_any:
  Prediction: 1 | Actual: 1 | ✓
  Probability: 79.19%

Governance:
  Prediction: 1 | Actual: 1 | ✓
  Probability: 91.71%

Social:
  Prediction: 1 | Actual: 1 | ✓
  Probability: 94.62%

Environmental:
  Prediction: 1 | Actual: 1 | ✓
  Probability: 91.03%

Climate_change:
  Prediction: 1 | Actual: 1 | ✓
  Probability: 92.26%


Accuracy: 5/5 (100%)
